# Grid Search CV

In [ ]:
import numpy as np
import pandas as pd

# Forecasting model, e.g. ARIMA
from sktime.forecasting.arima import ARIMA
# Modules for cross-validation
from sktime.split import temporal_train_test_split, ExpandingWindowSplitter, SlidingWindowSplitter, SingleWindowSplitter
from sktime.performance_metrics.forecasting import MeanSquaredError, MeanAbsoluteError, MeanAbsolutePercentageError # Metrics MSE, MAE, MAPE
# Grid search for optimal hyperparameters
from sktime.forecasting.model_selection import ForecastingGridSearchCV

import pandas_datareader.data as web

# visualization settings
import matplotlib.pyplot as plt

# Do not show Warnings
import warnings
warnings.simplefilter(action='ignore', category=Warning)
# Do not show ValueWarning, ConvergenceWarning from statsmodels
from statsmodels.tools.sm_exceptions import ValueWarning, ConvergenceWarning
warnings.simplefilter('ignore', category=ValueWarning)
warnings.simplefilter('ignore', category=ConvergenceWarning)

Load monthly data from the [`FRED`](https://fred.stlouisfed.org/) database on Market Yield on U.S. Treasury Securities at 10-Year Constant Maturity (Symbol [`GS10`](https://fred.stlouisfed.org/series/GS10)) from 2000-01-01 to present and create a dataframe `y`

In [ ]:
y = web.DataReader(name='GS10', data_source='fred', start='2000-01')
y.index = y.index.to_period(freq='M')

Grid search for the optimal (hyper)parameters of the model

In [ ]:
# Specify the forecasting method
forecaster = ARIMA()

# set up the cross-validation splitter
cv_strategy = ExpandingWindowSplitter(fh=np.arange(1, 6), initial_window=100, step_length=1)

# Define the grid of model parameter values as a dictionary
# we will vary the order and trend parameters of ARIMA
# Important: all possible combinations will be iterated over!
param_grid = {'order':[(0,1,0), (1,1,0), (0,1,1)], 'trend': ['c', 'n']}

# initialize the metric
metric = MeanSquaredError(square_root=False)

# Grid search
gscv = ForecastingGridSearchCV(forecaster=forecaster, param_grid=param_grid, cv=cv_strategy, scoring=metric)

gscv.fit(y)

In [ ]:
# Parameters of the optimal model
gscv.get_fitted_params()['best_forecaster']